In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
from time import perf_counter
from scipy.stats import norm, qmc

root = Path.home() / "Desktop/Chooser_Option_Research"
data_dir = root / "02_Data/processed"
fig_dir = root / "04_Reports/weekly/figures"

data_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

def bsm_option(S, K, T, r, q, sigma, kind):
    d1 = (np.log(S / K) + (r-q+0.5*sigma**2)*T) / (
        sigma * np.sqrt(T)
    )
    d2 = d1 - sigma * np.sqrt(T)

    if kind == "call":
        return (
            S*np.exp(-q*T)*norm.cdf(d1)
            - K*np.exp(-r*T)*norm.cdf(d2)
        )

    return (
        K*np.exp(-r*T)*norm.cdf(-d2)
        - S*np.exp(-q*T)*norm.cdf(-d1)
    )

def bsm_chooser(S, K, T1, T2, r1, r2, q, sigma):
    gap = T2 - T1
    forward = (r2*T2 - r1*T1) / gap

    call = bsm_option(
        S, K, T2, r2, q, sigma, "call"
    )

    adjusted_put = bsm_option(
        S*np.exp(-q*gap),
        K*np.exp(-forward*gap),
        T1, r1, q, sigma, "put"
    )

    return call + adjusted_put

In [7]:
def heston_chooser_cv(p, n_pairs=2048, n_steps=252, seed=42):
    fields = [
        "S", "K", "T1", "T2", "r1", "r2",
        "q", "v0", "theta", "kappa", "xi", "rho"
    ]

    x = {k: float(p[k]) for k in fields}

    S, K = x["S"], x["K"]
    T1, T2 = x["T1"], x["T2"]
    r1, r2, q = x["r1"], x["r2"], x["q"]
    v0, theta = x["v0"], x["theta"]
    kappa, xi, rho = x["kappa"], x["xi"], x["rho"]

    assert 0 < T1 < T2
    assert S > 0 and K > 0
    assert v0 > 0 and theta > 0 and kappa > 0
    assert xi >= 0 and -1 <= rho <= 1
    assert n_pairs >= 2 and n_steps >= 2

    gap = T2 - T1
    forward = (r2*T2 - r1*T1) / gap

    # Exactly include the choice date in the time grid
    n1 = max(1, min(n_steps-1, round(n_steps*T1/T2)))
    n2 = n_steps - n1

    rng = np.random.default_rng(seed)
    n = 2*n_pairs

    log_h = np.full(n, np.log(S))
    log_g = np.full(n, np.log(S))
    v = np.full(n, v0)

    for block, steps, dt, rate in [
        (1, n1, T1/n1, r1),
        (2, n2, gap/n2, forward)
    ]:
        for _ in range(steps):
            z = rng.standard_normal(n_pairs)
            e = rng.standard_normal(n_pairs)

            # Antithetic correlated Brownian shocks
            z1 = np.concatenate([z, -z])
            correlated = rho*z + np.sqrt(1-rho**2)*e
            z2 = np.concatenate([correlated, -correlated])

            vp = np.maximum(v, 0.0)

            log_h += (
                (rate-q-0.5*vp)*dt
                + np.sqrt(vp*dt)*z1
            )

            # GBM control path, using the SAME stock shocks
            log_g += (
                (rate-q-0.5*v0)*dt
                + np.sqrt(v0*dt)*z1
            )

            v += (
                kappa*(theta-vp)*dt
                + xi*np.sqrt(vp*dt)*z2
            )

        if block == 1:
            sh1 = np.exp(log_h.copy())
            sg1 = np.exp(log_g.copy())

    sh2 = np.exp(log_h)
    sg2 = np.exp(log_g)

    adjusted_strike = K*np.exp(-forward*gap)
    dividend_factor = np.exp(-q*gap)

    h_payoff = (
        np.exp(-r2*T2)*np.maximum(sh2-K, 0)
        + np.exp(-r1*T1)*np.maximum(
            adjusted_strike-dividend_factor*sh1, 0
        )
    )
    raw_pairs = 0.5 * (
    h_payoff[:n_pairs]
    + h_payoff[n_pairs:]
)

    raw_price = raw_pairs.mean()
    raw_se = raw_pairs.std(ddof=1) / np.sqrt(n_pairs)

    g_payoff = (
        np.exp(-r2*T2)*np.maximum(sg2-K, 0)
        + np.exp(-r1*T1)*np.maximum(
            adjusted_strike-dividend_factor*sg1, 0
        )
    )

    # Estimate Heston minus GBM using antithetic pairs
    differences = h_payoff - g_payoff
    pair_diff = (
        differences[:n_pairs] + differences[n_pairs:]
    ) / 2

    bsm = bsm_chooser(
        S, K, T1, T2, r1, r2, q, np.sqrt(v0)
    )

    price = bsm + pair_diff.mean()
    se = pair_diff.std(ddof=1) / np.sqrt(n_pairs)

    # Independent raw Monte Carlo estimate for validation
    raw_pairs = 0.5 * (
    h_payoff[:n_pairs] + h_payoff[n_pairs:]
)

    return {
    "price": float(price),
    "se": float(se),
    "bsm": float(bsm),
    "raw_price": float(raw_price),
    "raw_se": float(raw_se)
}

In [8]:
# W06 — explicit reference scenario; no hidden Kernel state
baseline = {
    "S": 239.7100067,
    "K": 239.7100067,
    "T1": 0.5,
    "T2": 1.0,
    "r1": 0.0424,
    "r2": 0.0416,
    "q": 0.01918985387,
    "v0": 0.1866672202**2,
    "theta": 0.1866672202**2,
    "kappa": 2.0,
    "xi": 0.35,
    "rho": -0.6,
}

In [9]:
bounds = {
    "S": (170, 300),
    "moneyness": (0.80, 1.20),   # K / S
    "T1": (0.20, 0.75),
    "gap": (0.30, 0.80),         # T2 - T1
    "r1": (0.01, 0.06),
    "forward": (0.01, 0.065),
    "q": (0.00, 0.035),
    "sig0": (0.13, 0.32),
    "sigbar": (0.13, 0.32),
    "kappa": (0.7, 3.5),
    "xi": (0.08, 0.50),
    "rho": (-0.85, -0.15)
}

def sample_cases(n, seed, split, stress=False):
    ranges = dict(bounds)

    if stress:
        ranges["moneyness"] = (1.20, 1.35)
        ranges["xi"] = (0.50, 0.65)

    keys = list(ranges)

    sampler = qmc.LatinHypercube(
        d=len(keys), seed=seed
    )
    sample = sampler.random(n)

    frame = pd.DataFrame({
        key: lo + (hi-lo)*sample[:, i]
        for i, (key, (lo, hi))
        in enumerate(ranges.items())
    })

    frame["K"] = frame["S"]*frame["moneyness"]
    frame["T2"] = frame["T1"]+frame["gap"]

    frame["r2"] = (
        frame["r1"]*frame["T1"]
        + frame["forward"]*frame["gap"]
    ) / frame["T2"]

    frame["v0"] = frame["sig0"]**2
    frame["theta"] = frame["sigbar"]**2
    frame["split"] = split

    return frame

cases = pd.concat([
    sample_cases(600, 101, "train"),
    sample_cases(150, 102, "validation"),
    sample_cases(150, 103, "test"),
    sample_cases(100, 104, "stress", stress=True)
], ignore_index=True)

cases["case_id"] = np.arange(len(cases))

print(cases["split"].value_counts())
display(cases.head())

# W06 Step 3 — Independent GBM Limit Validation

gbm_check = heston_chooser_cv(
    {**baseline, "xi": 0.0},
    n_pairs=8192,
    seed=42
)

difference = gbm_check["raw_price"] - gbm_check["bsm"]

print("Raw GBM MC:", gbm_check["raw_price"])
print("BSM Exact:", gbm_check["bsm"])
print("Difference / SE:", difference / gbm_check["raw_se"])

assert abs(difference) < 4 * gbm_check["raw_se"]

print("Independent GBM consistency check: PASSED")

# Heston baseline for comparison with W04
heston_check = heston_chooser_cv(
    baseline,
    n_pairs=8192,
    seed=42
)

print("Heston Benchmark:", heston_check)

split
train         600
validation    150
test          150
stress        100
Name: count, dtype: int64


,S,moneyness,T1,gap,r1,forward,q,sig0,sigbar,kappa,xi,rho,K,T2,r2,v0,theta,split,case_id
0,213.345568,1.123760,0.451364,0.352007,0.042142,0.012482,0.009516,0.195435,0.159775,3.078952,0.341936,-0.445961,239.749298,0.803371,0.029146,0.038195,0.025528,train,0
1,215.181270,0.957313,0.373346,0.467022,0.010051,0.018309,0.008070,0.253390,0.168473,1.944269,0.298361,-0.728959,205.995784,0.840368,0.014640,0.064207,0.028383,train,1
2,292.017791,1.058121,0.332305,0.300441,0.058836,0.025323,0.018769,0.173837,0.199681,0.962334,0.414341,-0.625382,308.990225,0.632746,0.042923,0.030219,0.039872,train,2
3,187.871138,1.056065,0.460595,0.521335,0.027878,0.056008,0.010471,0.182494,0.170187,1.142907,0.393449,-0.500816,198.404209,0.981930,0.042813,0.033304,0.028964,train,3
4,220.227844,0.976954,0.628448,0.364195,0.031045,0.046687,0.029690,0.192496,0.276306,1.833779,0.133965,-0.199182,215.152563,0.992642,0.036784,0.037055,0.076345,train,4


Raw GBM MC: 30.218581793195188
BSM Exact: 29.829017837147944
Difference / SE: 1.6492605185159086
Independent GBM consistency check: PASSED
Heston Benchmark: {'price': 28.868229822300087, 'se': 0.10650221823731, 'bsm': 29.829017837147944, 'raw_price': 29.257793778347327, 'raw_se': 0.18673963829601206}


In [10]:
records = []

for i, row in cases.iterrows():
    n_pairs = 2048 if row["split"] == "train" else 8192

    start = perf_counter()

    result = heston_chooser_cv(
        row,
        n_pairs=n_pairs,
        n_steps=252,
        seed=110000 + int(row["case_id"])
    )

    elapsed = perf_counter() - start

    records.append({
        "heston_price": result["price"],
        "mc_se": result["se"],
        "bsm_price": result["bsm"],
        "raw_mc_price": result["raw_price"],
        "mc_seconds": elapsed,
        "n_paths": 2*n_pairs,
        "n_steps": 252
    })

    if (i + 1) % 100 == 0:
        print(f"Generated {i+1}/{len(cases)} scenarios")

dataset = pd.concat([
    cases,
    pd.DataFrame(records)
], axis=1)

dataset["target_delta_norm"] = (
    dataset["heston_price"] - dataset["bsm_price"]
) / dataset["S"]

dataset["target_price_norm"] = (
    dataset["heston_price"] / dataset["S"]
)

dataset["feller_met"] = (
    2*dataset["kappa"]*dataset["theta"]
    >= dataset["xi"]**2
)

assert dataset["heston_price"].notna().all()
assert np.isfinite(dataset["heston_price"]).all()
assert (dataset["heston_price"] > 0).all()
assert (dataset["mc_se"] >= 0).all()

path = data_dir / "W06_Synthetic_Heston_Dataset.csv"
dataset.to_csv(path, index=False)

print("Saved:", path)
print("Shape:", dataset.shape)

display(
    dataset.groupby("split")[
        ["heston_price", "mc_se"]
    ].agg(["mean", "median"]).round(4)
)

Generated 100/1000 scenarios
Generated 200/1000 scenarios
Generated 300/1000 scenarios
Generated 400/1000 scenarios
Generated 500/1000 scenarios
Generated 600/1000 scenarios
Generated 700/1000 scenarios
Generated 800/1000 scenarios
Generated 900/1000 scenarios
Generated 1000/1000 scenarios
Saved: /Users/sienn/Desktop/Chooser_Option_Research/02_Data/processed/W06_Synthetic_Heston_Dataset.csv
Shape: (1000, 29)


heston_price            mc_se        
                   mean   median    mean  median
split                                           
stress          62.3441  61.0456  0.1697  0.1597
test            41.3039  39.9362  0.1165  0.1016
train           40.8475  40.2935  0.2306  0.2096
validation      41.0392  40.5088  0.1144  0.0995